# Lork Colab Quickstart: JSON Sentiment LoRA Mini-Project

Install `lork` cleanly on a Colab GPU runtime (removing Colab's incompatible `torchao==0.10.0`), run the async server with `lork serve`, train with prompt-masked `weights` and 2-microbatch gradient accumulation, test `stop` sequences, checkpoint with `save_state`/`load_state`, and export a standalone PEFT adapter with `export_adapter`.

In [ ]:
!pip install -q "lork[server]"
!pip uninstall -y -q torchao

In [ ]:
import os
import subprocess
import time

import httpx

os.environ["LORK_API_KEYS"] = "colab-secret-key"
os.environ["LORK_DEVICE"] = "cuda"
os.environ["LORK_HOST"] = "127.0.0.1"
os.environ["LORK_CHECKPOINT_ROOT"] = "/content/lork_checkpoints"

server_proc = subprocess.Popen(
    ["lork", "serve", "--host", "127.0.0.1", "--port", "8000"],
    env=os.environ.copy(),
)

with httpx.Client(base_url="http://127.0.0.1:8000", timeout=2.0) as probe:
    for _ in range(60):
        try:
            if probe.get("/v1/healthz").status_code == 200:
                print("lork server ready!")
                break
        except httpx.RequestError:
            pass
        time.sleep(0.5)

In [ ]:
from transformers import AutoTokenizer

from lork.client import OptimParams, SampleParams, ServiceClient

MODEL = "Qwen/Qwen2.5-0.5B"
tok = AutoTokenizer.from_pretrained(MODEL)
PAD_ID = tok.eos_token_id

TRAIN_EXAMPLES = [
    ("Loved the battery life and bright screen!", '{"sentiment": "positive"}'),
    ("Arrived cracked and customer support ignored me.", '{"sentiment": "negative"}'),
    ("It works fine, nothing special.", '{"sentiment": "neutral"}'),
    ("Best headphones I have ever bought.", '{"sentiment": "positive"}'),
    ("Stopped working after two days. Total waste.", '{"sentiment": "negative"}'),
    ("Standard quality for the price.", '{"sentiment": "neutral"}'),
]

def make_batch(pairs):
    seqs, tgts, wgts = [], [], []
    for review, target_json in pairs:
        prompt_ids = tok(f"Review: {review}\nJSON: ", add_special_tokens=False)["input_ids"]
        answer_ids = tok(target_json + "\n", add_special_tokens=False)["input_ids"]
        full = prompt_ids + answer_ids
        seqs.append(full[:-1])
        tgts.append(full[1:])
        wgts.append([0.0] * (len(prompt_ids) - 1) + [1.0] * len(answer_ids))
    max_len = max(len(x) for x in seqs)
    for i in range(len(seqs)):
        pad = max_len - len(seqs[i])
        seqs[i] += [PAD_ID] * pad
        tgts[i] += [PAD_ID] * pad
        wgts[i] += [0.0] * pad
    return {"input_ids": seqs, "target_ids": tgts, "weights": wgts}

micro_batches = [make_batch(TRAIN_EXAMPLES[:3]), make_batch(TRAIN_EXAMPLES[3:])]
test_prompts = [
    "Review: This keyboard feels amazing!\nJSON: ",
    "Review: Completely broken out of the box.\nJSON: ",
    "Review: Average product, does what it says.\nJSON: ",
]

with (
    ServiceClient("http://127.0.0.1:8000", "colab-secret-key", timeout=120.0) as client,
    client.create_training_client(
        MODEL, 8, 16.0, ["q_proj", "k_proj", "v_proj", "o_proj"], device="cuda"
    ) as tc,
):
    for _step in range(25):
        losses = [
            tc.forward_backward(mb, return_logprobs=False).result().loss
            for mb in micro_batches
        ]
        opt = tc.optim_step(OptimParams(lr=5e-4, weight_decay=0.01)).result()
        if opt.step % 5 == 0 or opt.step == 1:
            print(f"step={opt.step:02d} mean_loss={sum(losses) / len(losses):.4f}")

    tc.save_state("sentiment_ckpt").result()
    tc.load_state("sentiment_ckpt").result()
    export = tc.export_adapter("sentiment_peft").result()
    print("Exported PEFT adapter:", export.path, export.files)

    out = tc.sample(
        test_prompts,
        SampleParams(max_tokens=24, temperature=0.0, stop=["\n"]),
    ).result()
    for p, comp, reason in zip(test_prompts, out.completions, out.stop_reasons, strict=True):
        print(f"{p.splitlines()[0]} -> {comp!r} (stop={reason})")

In [ ]:
server_proc.terminate()
server_proc.wait(timeout=10)